# Data-Driven Quadratic Metric Learning via Semidefinite Programming

## Abstract
In quantitative research, defining an appropriate distance metric between multivariate financial features is critical for clustering and regime identification. This research note outlines a robust methodology to learn a quadratic pseudo-metric (Mahalanobis-like distance) directly from noisy pairwise distance measurements.

## Problem Formulation
We aim to learn a symmetric positive semidefinite matrix $P \in \mathbf{S}^n_+$ characterizing a quadratic pseudo-metric:
$$d(x,y) = \sqrt{(x-y)^T P (x-y)}$$

The non-linear Mean Squared Error (MSE) objective is recast into a tractable conic program constrained by the positive semidefinite (PSD) cone and rotated second-order cones (RSOCP):

Minimize:
$$\frac{1}{N}\sum_{i=1}^N (d_i^2 - 2d_i t_i + w_i)$$

Subject to:
$$P \in \mathbf{S}^n_+$$
$$w_i = (x_i - y_i)^T P (x_i - y_i), \quad i=1, \dots, N$$
$$t_i \ge 0, \quad i=1, \dots, N$$
$$(w_i, 1, t_i) \in \mathbf{Q}_r^3, \quad i=1, \dots, N$$

In [2]:
import numpy as np
from scipy import linalg as la

from src.metric_learning import MahalanobisMetricLearner

# 1. Generate Synthetic Data
np.random.seed(5680)
n, N = 5, 100

# True underlying metric P
P_true = np.random.randn(n, n)
P_true = P_true.dot(P_true.T) + np.identity(n)
sqrtP = la.sqrtm(P_true)

# Training Set
x_train = np.random.randn(N, n)
y_train = np.random.randn(N, n)
d_train = np.linalg.norm(sqrtP.dot((x_train - y_train).T), axis=0)
d_train = np.maximum(d_train + np.random.randn(N), 0)

# Test Set
N_test = 10
x_test = np.random.randn(N_test, n)
y_test = np.random.randn(N_test, n)
d_test = np.linalg.norm(sqrtP.dot((x_test - y_test).T), axis=0)
d_test = np.maximum(d_test + np.random.randn(N_test), 0)

In [3]:
# 2. Metric Learning via RSOCP/SDP Pipeline
learner = MahalanobisMetricLearner()
learner.fit(x_train, y_train, d_train)

# 3. Evaluate Out-of-Sample Performance
train_mse = learner.score(x_train, y_train, d_train)
test_mse = learner.score(x_test, y_test, d_test)

print(f"Optimal Train MSE (In-Sample): {train_mse:.4f}")
print(f"Test MSE (Out-of-Sample): {test_mse:.4f}")

Optimal Train MSE (In-Sample): 0.7203
Test MSE (Out-of-Sample): 0.9162
